In [ ]:
# ══════════════════════════════════════════════════════════
# SECTION 04 - EXPLORATORY DATA ANALYSIS
# Bangkok Airbnb Market Intelligence
# ══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Plot styling ───────────────────────────────────────────
plt.rcParams['figure.figsize']  = (12, 6)
plt.rcParams['font.size']       = 12
plt.rcParams['axes.spines.top']    = False
plt.rcParams['axes.spines.right']  = False
sns.set_palette("husl")

# ── Load master table ──────────────────────────────────────
CLEAN_PATH = '../data/bangkok/cleaned/'
RAW_PATH   = '../data/bangkok/'

master   = pd.read_csv(CLEAN_PATH + 'listings_master.csv')
nb_agg   = pd.read_csv(CLEAN_PATH + 'neighbourhood_aggregates.csv')
calendar = pd.read_csv(CLEAN_PATH + 'calendar_clean.csv')
reviews  = pd.read_csv(CLEAN_PATH + 'reviews_clean.csv')

# Parse dates
master['host_since']   = pd.to_datetime(master['host_since'],   errors='coerce')
master['first_review'] = pd.to_datetime(master['first_review'], errors='coerce')
master['last_review']  = pd.to_datetime(master['last_review'],  errors='coerce')
calendar['date']       = pd.to_datetime(calendar['date'],       errors='coerce')
reviews['date']        = pd.to_datetime(reviews['date'],        errors='coerce')

print("All datasets loaded!")
print(f"master:    {master.shape}")
print(f"nb_agg:    {nb_agg.shape}")
print(f"calendar:  {calendar.shape}")
print(f"reviews:   {reviews.shape}")

In [ ]:
# ══════════════════════════════════════════════════════════
# 4.1 SUMMARY STATISTICS & DISTRIBUTIONS
# ══════════════════════════════════════════════════════════

import os
PLOTS_DIR = '../reports/plots/'
os.makedirs(PLOTS_DIR, exist_ok=True)

# ── PLOT 1: Price Distribution by Room Type ────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Filter reasonable prices
price_data = master[
    (master['price_numeric'] > 0) & 
    (master['price_numeric'] <= 10000)
].copy()

# Left: Price distribution overall
axes[0].hist(price_data['price_numeric'], bins=50, 
             color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(price_data['price_numeric'].median(), 
                color='red', linestyle='--', linewidth=2,
                label=f"Median: ฿{price_data['price_numeric'].median():,.0f}")
axes[0].axvline(price_data['price_numeric'].mean(), 
                color='orange', linestyle='--', linewidth=2,
                label=f"Mean: ฿{price_data['price_numeric'].mean():,.0f}")
axes[0].set_title('Figure 1a: Price Distribution (≤฿10,000)', fontweight='bold')
axes[0].set_xlabel('Price per Night (THB)')
axes[0].set_ylabel('Number of Listings')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'฿{x:,.0f}'))

# Right: Box plot by room type
room_order = ['Entire home/apt', 'Private room', 'Hotel room', 'Shared room']
room_data  = [price_data[price_data['room_type'] == r]['price_numeric'].values 
              for r in room_order]
bp = axes[1].boxplot(room_data, tick_labels=room_order, patch_artist=True,
                     medianprops=dict(color='red', linewidth=2))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_title('Figure 1b: Price by Room Type', fontweight='bold')
axes[1].set_xlabel('Room Type')
axes[1].set_ylabel('Price per Night (THB)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'฿{x:,.0f}'))
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(PLOTS_DIR + 'fig01_price_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
BUSINESS INTERPRETATION - Figure 1:
The price distribution is strongly right-skewed, with most Bangkok listings 
priced between ฿500-฿3,000 per night. The median (฿1,379) is significantly 
lower than the mean (฿2,529), indicating a small number of luxury listings 
pulling the average up. Entire homes command the highest prices and widest 
price range, while shared rooms are tightly clustered at the budget end.

BUSINESS ACTION: Hosts pricing above ฿5,000 are in a thin luxury segment 
with limited competition but also limited demand. New hosts should target 
the ฿1,000-฿2,500 range where most bookings occur.
""")

In [ ]:
# ── PLOT 2: Top 20 Neighbourhoods by Median Price ─────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Prepare data
nb_price = master.groupby('neighbourhood_standard').agg(
    median_price  = ('price_numeric', 'median'),
    listing_count = ('id', 'count'),
    avg_occupancy = ('occupancy_rate', 'mean')
).reset_index().dropna(subset=['median_price'])

nb_price = nb_price.sort_values('median_price', ascending=False).head(20)

# Left: Bar chart - median price by neighbourhood
colors = ['#d32f2f' if i < 5 else '#1976d2' 
          for i in range(len(nb_price))]
axes[0].barh(nb_price['neighbourhood_standard'], 
             nb_price['median_price'], color=colors, alpha=0.8)
axes[0].axvline(master['price_numeric'].median(), 
                color='black', linestyle='--', linewidth=1.5,
                label=f"City median: ฿{master['price_numeric'].median():,.0f}")
axes[0].set_title('Figure 2a: Median Price by Neighbourhood (Top 20)', 
                  fontweight='bold')
axes[0].set_xlabel('Median Price per Night (THB)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'฿{x:,.0f}'))
axes[0].invert_yaxis()

# Right: Scatter - listing count vs median price
scatter_data = master.groupby('neighbourhood_standard').agg(
    median_price  = ('price_numeric', 'median'),
    listing_count = ('id', 'count'),
    avg_occupancy = ('occupancy_rate', 'mean')
).reset_index().dropna()

sc = axes[1].scatter(
    scatter_data['listing_count'],
    scatter_data['median_price'],
    c=scatter_data['avg_occupancy'],
    cmap='RdYlGn', s=100, alpha=0.8, edgecolors='gray'
)
plt.colorbar(sc, ax=axes[1], label='Avg Occupancy Rate (%)')

# Label top neighbourhoods
for _, row in scatter_data.nlargest(5, 'listing_count').iterrows():
    axes[1].annotate(row['neighbourhood_standard'],
                    (row['listing_count'], row['median_price']),
                    textcoords="offset points", xytext=(5, 5), fontsize=9)

axes[1].set_title('Figure 2b: Listing Count vs Price\n(color = occupancy)', 
                  fontweight='bold')
axes[1].set_xlabel('Number of Listings')
axes[1].set_ylabel('Median Price per Night (THB)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'฿{x:,.0f}'))

plt.tight_layout()
plt.savefig(PLOTS_DIR + 'fig02_neighbourhood_pricing.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
BUSINESS INTERPRETATION - Figure 2:
Premium neighbourhoods like Parthum Wan and Bang Rak command median prices 
significantly above the city median of ฿1,379. These areas correspond to 
Bangkok's CBD and luxury hotel districts. Interestingly, high listing count 
does not always mean high price — Vadhana has the most listings but moderate 
pricing, suggesting a competitive, volume-driven market. Green dots (high 
occupancy) tend to cluster in mid-price neighbourhoods, indicating that 
value-for-money listings attract more bookings than premium ones.

BUSINESS ACTION: Investors should target mid-tier neighbourhoods with high 
occupancy (green) rather than chasing premium prices in luxury areas.
""")

In [ ]:
# ── PLOT 3: Seasonal Trends ────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Monthly occupancy
calendar['month'] = calendar['date'].dt.month
calendar['month_name'] = calendar['date'].dt.strftime('%b')
calendar['is_booked'] = calendar['available'] == 'f'

monthly = calendar.groupby(['month', 'month_name']).agg(
    total_days  = ('available', 'count'),
    booked_days = ('is_booked', 'sum')
).reset_index()
monthly['occupancy_pct'] = (monthly['booked_days'] / monthly['total_days'] * 100).round(2)
monthly = monthly.sort_values('month')

# Top: Monthly occupancy rate
colors = ['#d32f2f' if x >= 40 else '#1976d2' for x in monthly['occupancy_pct']]
bars = axes[0].bar(monthly['month_name'], monthly['occupancy_pct'], 
                   color=colors, alpha=0.85, edgecolor='white')
axes[0].axhline(monthly['occupancy_pct'].mean(), color='black', 
                linestyle='--', linewidth=1.5,
                label=f"Annual avg: {monthly['occupancy_pct'].mean():.1f}%")
for bar, val in zip(bars, monthly['occupancy_pct']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Figure 3a: Monthly Occupancy Rate - Bangkok Airbnb', 
                  fontweight='bold')
axes[0].set_ylabel('Occupancy Rate (%)')
axes[0].set_ylim(0, 55)
axes[0].legend()

# Bottom: Review volume by year (demand proxy)
reviews['year'] = reviews['date'].dt.year
yearly_reviews = reviews.groupby('year').size().reset_index(name='review_count')
yearly_reviews = yearly_reviews[yearly_reviews['year'] >= 2015]

axes[1].plot(yearly_reviews['year'], yearly_reviews['review_count'], 
             marker='o', linewidth=2.5, markersize=8, color='#1976d2')
axes[1].fill_between(yearly_reviews['year'], yearly_reviews['review_count'], 
                     alpha=0.2, color='#1976d2')

# Annotate COVID dip
axes[1].annotate('COVID-19\nimpact', xy=(2020, 
                  yearly_reviews[yearly_reviews['year']==2020]['review_count'].values[0]),
                 xytext=(2018, 40000),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 color='red', fontsize=10)

axes[1].set_title('Figure 3b: Annual Review Volume 2015-2025 (Demand Proxy)', 
                  fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Reviews')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x:,.0f}'))
axes[1].set_xticks(yearly_reviews['year'])

plt.tight_layout()
plt.savefig(PLOTS_DIR + 'fig03_seasonal_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
BUSINESS INTERPRETATION - Figure 3:
Bangkok Airbnb shows clear seasonality with peak occupancy in July-September 
(40%+), coinciding with the international summer travel season. The low season 
falls in February-March (~24%). The annual review volume chart reveals strong 
market growth from 2015-2019, a severe COVID-19 crash in 2020-2021, and a 
strong recovery by 2023-2024.

BUSINESS ACTION: Hosts should implement dynamic pricing — raising rates 20-30% 
during July-September peak season and offering discounts in Feb-March to 
maintain occupancy. Revenue managers should plan inventory around the mid-year 
peak.
""")

In [ ]:
# ── PLOT 3: Seasonal Trends (using DuckDB for efficiency) ──
import duckdb

con = duckdb.connect('../data/bangkok/airbnb_bangkok.duckdb')

# Aggregate in DuckDB instead of pandas
monthly = con.execute("""
    SELECT
        MONTH(date)                    AS month,
        STRFTIME(date, '%b')           AS month_name,
        COUNT(*)                       AS total_days,
        SUM(CASE WHEN available = 'f' 
            THEN 1 ELSE 0 END)         AS booked_days
    FROM read_csv_auto('../data/bangkok/cleaned/calendar_clean.csv')
    WHERE date IS NOT NULL
    GROUP BY MONTH(date), STRFTIME(date, '%b')
    ORDER BY month
""").df()

monthly['occupancy_pct'] = (
    monthly['booked_days'] / monthly['total_days'] * 100
).round(2)

print(monthly)
con.close()

In [ ]:
# ── PLOT 3: Seasonal Trends ────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: Monthly occupancy rate
colors = ['#d32f2f' if x >= 40 else '#1976d2' for x in monthly['occupancy_pct']]
bars = axes[0].bar(monthly['month_name'], monthly['occupancy_pct'],
                   color=colors, alpha=0.85, edgecolor='white')
axes[0].axhline(monthly['occupancy_pct'].mean(), color='black',
                linestyle='--', linewidth=1.5,
                label=f"Annual avg: {monthly['occupancy_pct'].mean():.1f}%")
for bar, val in zip(bars, monthly['occupancy_pct']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Figure 3a: Monthly Occupancy Rate - Bangkok Airbnb',
                  fontweight='bold')
axes[0].set_ylabel('Occupancy Rate (%)')
axes[0].set_ylim(0, 55)
axes[0].legend()

# Bottom: Review volume by year (demand proxy)
reviews['year'] = reviews['date'].dt.year
yearly_reviews = reviews.groupby('year').size().reset_index(name='review_count')
yearly_reviews = yearly_reviews[yearly_reviews['year'] >= 2015]

axes[1].plot(yearly_reviews['year'], yearly_reviews['review_count'],
             marker='o', linewidth=2.5, markersize=8, color='#1976d2')
axes[1].fill_between(yearly_reviews['year'], yearly_reviews['review_count'],
                     alpha=0.2, color='#1976d2')

# Annotate COVID dip
covid_val = yearly_reviews[yearly_reviews['year']==2020]['review_count'].values
if len(covid_val) > 0:
    axes[1].annotate('COVID-19\nimpact',
                     xy=(2020, covid_val[0]),
                     xytext=(2018, 50000),
                     arrowprops=dict(arrowstyle='->', color='red'),
                     color='red', fontsize=10)

axes[1].set_title('Figure 3b: Annual Review Volume 2015-2025 (Demand Proxy)',
                  fontweight='bold')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Number of Reviews')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x:,.0f}'))
axes[1].set_xticks(yearly_reviews['year'])

plt.tight_layout()
plt.savefig(PLOTS_DIR + 'fig03_seasonal_trends.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
BUSINESS INTERPRETATION - Figure 3:
Bangkok Airbnb shows clear seasonality with peak occupancy in July-September 
(40%+), coinciding with the international summer travel season. The low season 
falls in February-March (~24%). The annual review volume chart reveals strong 
market growth from 2015-2019, a severe COVID-19 crash in 2020-2021, and a 
strong recovery by 2023-2025.

BUSINESS ACTION: Hosts should implement dynamic pricing — raising rates 20-30% 
during July-September peak season and offering discounts in Feb-March to 
maintain occupancy. Revenue managers should plan inventory around the mid-year 
peak.
""")

In [ ]:
# ── PLOT 4: Host & Supply Side Analysis ───────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── 4a: Host portfolio distribution ───────────────────────
host_counts = master.groupby('host_id')['id'].count().reset_index()
host_counts.columns = ['host_id', 'listing_count']
host_segments = pd.cut(host_counts['listing_count'],
                       bins=[0,1,5,10,50,1000],
                       labels=['1 listing','2-5','6-10','11-50','50+'])
segment_counts = host_segments.value_counts().sort_index()

axes[0,0].bar(segment_counts.index, segment_counts.values,
              color=['#4CAF50','#2196F3','#FF9800','#F44336','#9C27B0'],
              alpha=0.85, edgecolor='white')
for i, (idx, val) in enumerate(segment_counts.items()):
    axes[0,0].text(i, val + 20, f'{val:,}', ha='center', fontsize=10)
axes[0,0].set_title('Figure 4a: Host Portfolio Size Distribution',
                    fontweight='bold')
axes[0,0].set_xlabel('Number of Listings per Host')
axes[0,0].set_ylabel('Number of Hosts')

# ── 4b: Superhost vs non-superhost ────────────────────────
superhost_stats = master.groupby('host_is_superhost').agg(
    avg_price     = ('price_numeric', 'mean'),
    avg_occupancy = ('occupancy_rate', 'mean'),
    avg_rating    = ('review_scores_rating', lambda x: x[x>0].mean()),
    count         = ('id', 'count')
).reset_index()
superhost_stats['label'] = superhost_stats['host_is_superhost'].map(
    {'t': 'Superhost', 'f': 'Regular Host'}
)

metrics = ['avg_price', 'avg_occupancy', 'avg_rating']
labels  = ['Avg Price (THB)', 'Avg Occupancy (%)', 'Avg Rating']
x = np.arange(len(metrics))
width = 0.35

# Normalize for comparison
for i, (metric, label) in enumerate(zip(metrics, labels)):
    vals = superhost_stats[metric].values
    axes[0,1].bar(x + (0.2 if j==0 else -0.2),
                  vals, width,
                  label=superhost_stats['label'].values[j],
                  alpha=0.85) if False else None

# Simpler grouped bar
categories = ['Avg Price\n(THB/10)', 'Avg Occupancy\n(%)', 'Avg Rating\nx20']
regular = superhost_stats[superhost_stats['host_is_superhost']=='f']
super_h = superhost_stats[superhost_stats['host_is_superhost']=='t']

reg_vals = [
    regular['avg_price'].values[0]/10,
    regular['avg_occupancy'].values[0],
    regular['avg_rating'].values[0]*20
]
sup_vals = [
    super_h['avg_price'].values[0]/10,
    super_h['avg_occupancy'].values[0],
    super_h['avg_rating'].values[0]*20
]

x = np.arange(len(categories))
axes[0,1].bar(x - 0.2, reg_vals, 0.35, label='Regular Host',
              color='#2196F3', alpha=0.85)
axes[0,1].bar(x + 0.2, sup_vals, 0.35, label='Superhost',
              color='#FF9800', alpha=0.85)
axes[0,1].set_title('Figure 4b: Superhost vs Regular Host Performance',
                    fontweight='bold')
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(categories)
axes[0,1].legend()
axes[0,1].set_ylabel('Scaled Value')

# ── 4c: Host tenure vs price ──────────────────────────────
tenure_data = master[
    (master['host_tenure_years'].notna()) &
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 10000)
].copy()
tenure_data['tenure_group'] = pd.cut(tenure_data['host_tenure_years'],
    bins=[0,1,3,5,10,20],
    labels=['<1yr','1-3yr','3-5yr','5-10yr','10+yr'])

tenure_price = tenure_data.groupby('tenure_group', observed=True).agg(
    median_price  = ('price_numeric', 'median'),
    avg_occupancy = ('occupancy_rate', 'mean'),
    count         = ('id', 'count')
).reset_index()

axes[1,0].bar(tenure_price['tenure_group'].astype(str),
              tenure_price['median_price'],
              color='#1976d2', alpha=0.85, edgecolor='white')
ax2 = axes[1,0].twinx()
ax2.plot(tenure_price['tenure_group'].astype(str),
         tenure_price['avg_occupancy'],
         color='red', marker='o', linewidth=2, markersize=8,
         label='Avg Occupancy')
axes[1,0].set_title('Figure 4c: Host Tenure vs Price & Occupancy',
                    fontweight='bold')
axes[1,0].set_xlabel('Host Tenure')
axes[1,0].set_ylabel('Median Price (THB)', color='#1976d2')
ax2.set_ylabel('Avg Occupancy (%)', color='red')
ax2.legend(loc='upper right')

# ── 4d: Market concentration ──────────────────────────────
host_listings = master.groupby('host_id')['id'].count().sort_values(ascending=False)
cumulative_listings = host_listings.cumsum() / host_listings.sum() * 100
cumulative_hosts    = np.arange(1, len(host_listings)+1) / len(host_listings) * 100

axes[1,1].plot(cumulative_hosts, cumulative_listings,
               color='#1976d2', linewidth=2.5)
axes[1,1].plot([0,100],[0,100], 'k--', linewidth=1, label='Perfect equality')
axes[1,1].fill_between(cumulative_hosts, cumulative_listings,
                       cumulative_hosts, alpha=0.2, color='#1976d2',
                       label='Concentration area')

# Mark key points
for pct in [10, 20]:
    idx = np.searchsorted(cumulative_hosts, pct)
    y_val = cumulative_listings.iloc[idx]
    axes[1,1].annotate(f'Top {pct}% hosts\ncontrol {y_val:.0f}% listings',
                      xy=(pct, y_val), xytext=(pct+10, y_val-15),
                      arrowprops=dict(arrowstyle='->', color='red'),
                      fontsize=9, color='red')

axes[1,1].set_title('Figure 4d: Market Concentration (Lorenz Curve)',
                    fontweight='bold')
axes[1,1].set_xlabel('Cumulative % of Hosts')
axes[1,1].set_ylabel('Cumulative % of Listings')
axes[1,1].legend()

plt.tight_layout()
plt.savefig(PLOTS_DIR + 'fig04_host_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
BUSINESS INTERPRETATION - Figure 4:
Most hosts (majority) manage only 1 listing, indicating a market dominated 
by casual hosts. However, the Lorenz curve reveals significant concentration 
— a small number of professional hosts control a disproportionate share of 
listings. Superhosts charge less but achieve higher occupancy and ratings, 
suggesting that quality and reputation drive bookings more than price. 
Experienced hosts (10+ years) command higher prices, reflecting the value 
of platform reputation and optimization over time.

BUSINESS ACTION: New hosts should focus on achieving Superhost status rather 
than maximizing price. Property management companies should target the 
professional multi-listing segment where operational efficiency matters most.
""")

In [ ]:
# ── PLOT 5: Review & Rating Analysis ──────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── 5a: Review score distribution ─────────────────────────
rated = master[master['review_scores_rating'] > 0]['review_scores_rating']
axes[0,0].hist(rated, bins=30, color='#1976d2', 
               edgecolor='white', alpha=0.85)
axes[0,0].axvline(rated.mean(), color='red', linestyle='--',
                  linewidth=2, label=f'Mean: {rated.mean():.2f}')
axes[0,0].axvline(rated.median(), color='orange', linestyle='--',
                  linewidth=2, label=f'Median: {rated.median():.2f}')
axes[0,0].set_title('Figure 5a: Review Score Distribution', fontweight='bold')
axes[0,0].set_xlabel('Review Score (out of 5)')
axes[0,0].set_ylabel('Number of Listings')
axes[0,0].legend()

# ── 5b: Review scores by room type ────────────────────────
room_ratings = master[master['review_scores_rating'] > 0].groupby('room_type').agg(
    avg_rating = ('review_scores_rating', 'mean'),
    count      = ('id', 'count')
).reset_index().sort_values('avg_rating', ascending=True)

bars = axes[0,1].barh(room_ratings['room_type'], 
                      room_ratings['avg_rating'],
                      color='#4CAF50', alpha=0.85, edgecolor='white')
axes[0,1].axvline(rated.mean(), color='red', linestyle='--',
                  linewidth=1.5, label=f'Overall avg: {rated.mean():.2f}')
for bar, val in zip(bars, room_ratings['avg_rating']):
    axes[0,1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                  f'{val:.2f}', va='center', fontsize=11)
axes[0,1].set_title('Figure 5b: Average Rating by Room Type', fontweight='bold')
axes[0,1].set_xlabel('Average Review Score')
axes[0,1].set_xlim(4.0, 5.0)
axes[0,1].legend()

# ── 5c: Review count vs price ─────────────────────────────
sample = master[
    (master['price_numeric'] > 0) &
    (master['price_numeric'] <= 8000) &
    (master['number_of_reviews'] <= 300)
].sample(2000, random_state=42)

sc = axes[1,0].scatter(sample['number_of_reviews'],
                       sample['price_numeric'],
                       c=sample['review_scores_rating'].replace(-1, np.nan),
                       cmap='RdYlGn', alpha=0.5, s=20,
                       vmin=4.0, vmax=5.0)
plt.colorbar(sc, ax=axes[1,0], label='Review Score')
axes[1,0].set_title('Figure 5c: Review Count vs Price\n(color = rating)',
                    fontweight='bold')
axes[1,0].set_xlabel('Number of Reviews')
axes[1,0].set_ylabel('Price per Night (THB)')
axes[1,0].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'฿{x:,.0f}'))

# ── 5d: Review sub-dimensions ─────────────────────────────
sub_dims = {
    'Cleanliness':     'review_scores_cleanliness',
    'Location':        'review_scores_location',
    'Communication':   'review_scores_communication',
    'Check-in':        'review_scores_checkin',
    'Accuracy':        'review_scores_accuracy',
    'Value':           'review_scores_value'
}
avg_scores = {}
for label, col in sub_dims.items():
    if col in master.columns:
        vals = master[master[col] > 0][col]
        avg_scores[label] = vals.mean()

dims   = list(avg_scores.keys())
scores = list(avg_scores.values())
colors = ['#d32f2f' if s == min(scores) else 
          '#4CAF50' if s == max(scores) else '#1976d2' 
          for s in scores]

bars = axes[1,1].bar(dims, scores, color=colors, alpha=0.85, edgecolor='white')
axes[1,1].set_ylim(4.0, 5.0)
for bar, val in zip(bars, scores):
    axes[1,1].text(bar.get_x() + bar.get_width()/2, val + 0.005,
                  f'{val:.3f}', ha='center', va='bottom', fontsize=10)
axes[1,1].set_title('Figure 5d: Average Review Sub-Dimension Scores',
                    fontweight='bold')
axes[1,1].set_xlabel('Review Dimension')
axes[1,1].set_ylabel('Average Score (out of 5)')
axes[1,1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(PLOTS_DIR + 'fig05_review_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
BUSINESS INTERPRETATION - Figure 5:
Review scores are heavily left-skewed with most listings rated 4.5-5.0, 
suggesting rating inflation on the platform. Location scores are highest 
(guests are happy with Bangkok locations) while Value scores are lowest 
(guests feel some listings are overpriced relative to quality). Shared rooms 
achieve surprisingly competitive ratings despite lower prices. High review 
count listings tend to cluster in the ฿500-฿3,000 range, confirming that 
mid-market listings generate the most consistent booking activity.

BUSINESS ACTION: Hosts should focus on improving Value perception — either 
by adding amenities or adjusting pricing. Location is already a strength of 
Bangkok listings and should be highlighted in listing descriptions.
""")

In [ ]:
# Section 04 Summary
print("""
SECTION 04 - EDA COMPLETE
=========================
Figures created:
  Fig 1: Price distribution & room type comparison
  Fig 2: Neighbourhood pricing & occupancy scatter
  Fig 3: Seasonal occupancy & demand trends
  Fig 4: Host analysis & market concentration
  Fig 5: Review scores & sub-dimensions

Key findings:
  - Prices right-skewed, median ฿1,379, mean ฿1,825
  - Vadhana is Bangkok's top Airbnb neighbourhood
  - Peak season: July-September (40%+ occupancy)
  - COVID crash 2020-2021, strong recovery 2024
  - Top 10% hosts control 56% of listings
  - Superhosts earn higher occupancy despite lower prices
  - Value is the weakest review dimension (4.647)
  - Communication is strongest (4.781)
""")